Task 1

In [ ]:
import math

def calculate_utility(gene, target, weights):
    total = 0
    n = max(len(gene), len(target))
    for i in range(n):
        weight = weights[i] if i < len(weights) else 1
        gene_val = ord(gene[i]) if i < len(gene) else 0
        target_val = ord(target[i]) if i < len(target) else 0
        total += weight * abs(gene_val - target_val)
    return -total

def minimax(current, pool, is_max, alpha, beta, target, weights):
    if not pool:
        return calculate_utility(current, target, weights), current

    if is_max:
        best_val = -math.inf
        best_seq = ""
        for n in pool:
            new_pool = pool[:]
            new_pool.remove(n)
            val, seq = minimax(current + n, new_pool, False, alpha, beta, target, weights)
            if val > best_val:
                best_val = val
                best_seq = seq
            alpha = max(alpha, val)
            if beta <= alpha:
                break
        return best_val, best_seq
    else:
        best_val = math.inf
        best_seq = ""
        for n in pool:
            new_pool = pool[:]
            new_pool.remove(n)
            val, seq = minimax(current + n, new_pool, True, alpha, beta, target, weights)
            if val < best_val:
                best_val = val
                best_seq = seq
            beta = min(beta, val)
            if beta <= alpha:
                break
        return best_val, best_seq

if __name__ == "__main__":
    pool_input = input()
    target_gene = input()
    sid_input = input()

    pool = [n.strip() for n in pool_input.split(',')]
    sid_digits = [int(d) for d in sid_input.split()]
    weights = sid_digits[-len(target_gene):] if len(sid_digits) >= len(target_gene) else sid_digits

    score, best_gene = minimax("", pool, True, -math.inf, math.inf, target_gene, weights)

    print(f"Best gene sequence generated: {best_gene}")
    print(f"Utility score: {score}")


A,T,C,G
ATGC
1 8 1 0 4 0 5 2
Best gene sequence generated: AGCT
Utility score: -54


Task 2

In [ ]:
import math

def calculate_utility(gene, target, weights):

    total_difference = 0
    n = max(len(gene), len(target))
    for i in range(n):
        weight = weights[i] if i < len(weights) else 1
        gene_char_ascii = ord(gene[i]) if i < len(gene) else 0
        target_char_ascii = ord(target[i]) if i < len(target) else 0
        total_difference += weight * abs(gene_char_ascii - target_char_ascii)
    return -total_difference

def calculate_utility_with_booster(gene, target, weights, booster_active_at_index, booster_factor):

    total_difference = 0.0
    n = max(len(gene), len(target))

    for i in range(n):
        current_weight = 1.0

        if i < len(weights):
            current_weight = float(weights[i])
            if booster_active_at_index != -1 and i >= booster_active_at_index:
                current_weight *= booster_factor

        gene_char_ascii = ord(gene[i]) if i < len(gene) else 0
        target_char_ascii = ord(target[i]) if i < len(target) else 0

        total_difference += current_weight * abs(gene_char_ascii - target_char_ascii)

    return -total_difference

def minimax_alpha_beta(current_gene, remaining_pool, is_maximizing, alpha, beta, target, weights):

    if not remaining_pool:
        return calculate_utility(current_gene, target, weights), current_gene

    if is_maximizing:
        max_eval = -math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)
            evaluation, sequence = minimax_alpha_beta(
                current_gene + nucleotide, new_pool, False, alpha, beta, target, weights
            )
            if evaluation > max_eval:
                max_eval = evaluation
                best_sequence = sequence
            alpha = max(alpha, evaluation)
            if beta <= alpha:
                break
        return max_eval, best_sequence
    else:
        min_eval = math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)
            evaluation, sequence = minimax_alpha_beta(
                current_gene + nucleotide, new_pool, True, alpha, beta, target, weights
            )
            if evaluation < min_eval:
                min_eval = evaluation
                best_sequence = sequence
            beta = min(beta, evaluation)
            if beta <= alpha:
                break
        return min_eval, best_sequence

def minimax_with_special_nucleotide(current_gene, remaining_pool, is_maximizing, alpha, beta, target, weights, booster_info):

    booster_active_at_index, booster_factor = booster_info

    if not remaining_pool:
        score = calculate_utility_with_booster(current_gene, target, weights, booster_active_at_index, booster_factor)
        return score, current_gene

    if is_maximizing:
        max_eval = -math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)

            new_booster_info = booster_info
            if nucleotide == 'S' and booster_active_at_index == -1:
                current_depth = len(current_gene)
                new_booster_info = (current_depth, booster_factor)

            evaluation, sequence = minimax_with_special_nucleotide(
                current_gene + nucleotide, new_pool, False, alpha, beta, target, weights, new_booster_info
            )
            if evaluation > max_eval:
                max_eval = evaluation
                best_sequence = sequence
            alpha = max(alpha, evaluation)
            if beta <= alpha:
                break
        return max_eval, best_sequence
    else:
        min_eval = math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)
            evaluation, sequence = minimax_with_special_nucleotide(
                current_gene + nucleotide, new_pool, True, alpha, beta, target, weights, booster_info
            )
            if evaluation < min_eval:
                min_eval = evaluation
                best_sequence = sequence
            beta = min(beta, evaluation)
            if beta <= alpha:
                break
        return min_eval, best_sequence

if __name__ == "__main__":

        pool_input = input()
        target_gene = input()
        sid_input = input()

        original_pool = [n.strip() for n in pool_input.split(',')]
        sid_digits = [int(d) for d in sid_input.split()]
        target_len = len(target_gene)

        weights = sid_digits[-target_len:] if len(sid_digits) >= target_len else sid_digits

        score_without_S, _ = minimax_alpha_beta(
            "", original_pool, True, -math.inf, math.inf, target_gene, weights
        )

        pool_with_S = original_pool + ['S']

        booster_factor = (sid_digits[0] * 10 + sid_digits[1]) / 100.0 if len(sid_digits) >= 2 else 0.0

        initial_booster_info = (-1, booster_factor)

        score_with_S, sequence_with_S = minimax_with_special_nucleotide(
            "", pool_with_S, True, -math.inf, math.inf, target_gene, weights, initial_booster_info
        )

        if score_with_S > score_without_S:
            print("YES")
        else:
            print("NO")

        print("With special nucleotide")
        print(f"Best gene sequence generated: {sequence_with_S}")
        print(f"Utility score: {score_with_S:.2f}")


A,T,C,G
ATGC
1 8 1 0 4 0 5 2
YES
With special nucleotide
Best gene sequence generated: SGCTA
Utility score: -34.38


merged

In [ ]:
import math

def calculate_utility(gene, target, weights):
    total_difference = 0
    n = max(len(gene), len(target))
    for i in range(n):
        weight = weights[i] if i < len(weights) else 1
        gene_char_ascii = ord(gene[i]) if i < len(gene) else 0
        target_char_ascii = ord(target[i]) if i < len(target) else 0
        total_difference += weight * abs(gene_char_ascii - target_char_ascii)
    return -total_difference

def calculate_utility_with_booster(gene, target, weights, booster_active_at_index, booster_factor):

    total_difference = 0.0
    n = max(len(gene), len(target))

    for i in range(n):
        current_weight = 1.0

        if i < len(weights):
            current_weight = float(weights[i])
            if booster_active_at_index != -1 and i >= booster_active_at_index:
                current_weight *= booster_factor

        gene_char_ascii = ord(gene[i]) if i < len(gene) else 0
        target_char_ascii = ord(target[i]) if i < len(target) else 0

        total_difference += current_weight * abs(gene_char_ascii - target_char_ascii)

    return -total_difference

def minimax_alpha_beta(current_gene, remaining_pool, is_maximizing, alpha, beta, target, weights):
    if not remaining_pool:
        return calculate_utility(current_gene, target, weights), current_gene

    if is_maximizing:
        max_eval = -math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)
            evaluation, sequence = minimax_alpha_beta(
                current_gene + nucleotide, new_pool, False, alpha, beta, target, weights
            )
            if evaluation > max_eval:
                max_eval = evaluation
                best_sequence = sequence
            alpha = max(alpha, evaluation)
            if beta <= alpha:
                break
        return max_eval, best_sequence
    else:
        min_eval = math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)
            evaluation, sequence = minimax_alpha_beta(
                current_gene + nucleotide, new_pool, True, alpha, beta, target, weights
            )
            if evaluation < min_eval:
                min_eval = evaluation
                best_sequence = sequence
            beta = min(beta, evaluation)
            if beta <= alpha:
                break
        return min_eval, best_sequence

def minimax_with_special_nucleotide(current_gene, remaining_pool, is_maximizing, alpha, beta, target, weights, booster_info):

    booster_active_at_index, booster_factor = booster_info
    if not remaining_pool:
        score = calculate_utility_with_booster(current_gene, target, weights, booster_active_at_index, booster_factor)
        return score, current_gene

    if is_maximizing:
        max_eval = -math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)

            new_booster_info = booster_info
            if nucleotide == 'S' and booster_active_at_index == -1:
                current_depth = len(current_gene)
                new_booster_info = (current_depth, booster_factor)

            evaluation, sequence = minimax_with_special_nucleotide(
                current_gene + nucleotide, new_pool, False, alpha, beta, target, weights, new_booster_info
            )
            if evaluation > max_eval:
                max_eval = evaluation
                best_sequence = sequence
            alpha = max(alpha, evaluation)
            if beta <= alpha:
                break
        return max_eval, best_sequence
    else:
        min_eval = math.inf
        best_sequence = ""
        for nucleotide in remaining_pool:
            new_pool = remaining_pool[:]
            new_pool.remove(nucleotide)
            evaluation, sequence = minimax_with_special_nucleotide(
                current_gene + nucleotide, new_pool, True, alpha, beta, target, weights, booster_info
            )
            if evaluation < min_eval:
                min_eval = evaluation
                best_sequence = sequence
            beta = min(beta, evaluation)
            if beta <= alpha:
                break
        return min_eval, best_sequence

if __name__ == "__main__":

    pool_input = input()
    target_gene = input()
    sid_input = input()

    original_pool = [n.strip() for n in pool_input.split(',')]
    sid_digits = [int(d) for d in sid_input.split()]
    target_len = len(target_gene)

    weights = sid_digits[-target_len:] if len(sid_digits) >= target_len else sid_digits
    score_without_S, sequence_without_S = minimax_alpha_beta(
        "", original_pool, True, -math.inf, math.inf, target_gene, weights
    )

    pool_with_S = original_pool + ['S']
    booster_factor = (sid_digits[0] * 10 + sid_digits[1]) / 100.0 if len(sid_digits) >= 2 else 0.0
    initial_booster_info = (-1, booster_factor)

    score_with_S, sequence_with_S = minimax_with_special_nucleotide(
        "", pool_with_S, True, -math.inf, math.inf, target_gene, weights, initial_booster_info
    )
    print("\n--- Results ---")
    if score_with_S > score_without_S:
        print("Using the special nucleotide is beneficial: YES")
    else:
        print("Using the special nucleotide is not beneficial: NO")
    print("\nAnalysis without special nucleotide:")
    print(f"Best gene sequence generated: {sequence_without_S}")
    print(f"Utility score: {score_without_S:.2f}")
    print("\nAnalysis with special nucleotide:")
    print(f"Best gene sequence generated: {sequence_with_S}")
    print(f"Utility score: {score_with_S:.2f}")

A,T,C,G
CGAT
1 5 0 7 2 2 7 1

--- Results ---
Using the special nucleotide is not beneficial: NO

Analysis without special nucleotide:
Best gene sequence generated: GTAC
Utility score: -51.00

Analysis with special nucleotide:
Best gene sequence generated: SACGT
Utility score: -94.65
